In [2]:
import numpy as np
import pandas as pd

In [20]:
df = pd.read_csv("deliveries.csv").head()
df, df.shape

(   match_id  inning           batting_team                 bowling_team  over  \
 0    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
 1    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
 2    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
 3    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
 4    335982       1  Kolkata Knight Riders  Royal Challengers Bangalore     0   
 
    ball       batter   bowler  non_striker  batsman_runs  extra_runs  \
 0     1   SC Ganguly  P Kumar  BB McCullum           0.0         1.0   
 1     2  BB McCullum  P Kumar   SC Ganguly           0.0         0.0   
 2     3  BB McCullum  P Kumar   SC Ganguly           0.0         1.0   
 3     4  BB McCullum  P Kumar   SC Ganguly           0.0         0.0   
 4     5  BB McCullum  P Kumar   SC Ganguly           0.0         0.0   
 
    total_runs extras_type  is_wicket player_dismissed dismissal_k

In [21]:
df = pd.read_csv("matches.csv").head()
df, df.shape

(       id   season        city        date match_type player_of_match  \
 0  335982  2007/08   Bangalore  2008-04-18     League     BB McCullum   
 1  335983  2007/08  Chandigarh  2008-04-19     League      MEK Hussey   
 2  335984  2007/08       Delhi  2008-04-19     League     MF Maharoof   
 3  335985  2007/08      Mumbai  2008-04-20     League      MV Boucher   
 4  335986  2007/08     Kolkata  2008-04-20     League       DJ Hussey   
 
                                         venue                        team1  \
 0                       M Chinnaswamy Stadium  Royal Challengers Bangalore   
 1  Punjab Cricket Association Stadium, Mohali              Kings XI Punjab   
 2                            Feroz Shah Kotla             Delhi Daredevils   
 3                            Wankhede Stadium               Mumbai Indians   
 4                                Eden Gardens        Kolkata Knight Riders   
 
                          team2                  toss_winner toss_decision  \


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [9]:
deliveries = pd.read_csv("deliveries.csv")
matches = pd.read_csv("matches.csv")

In [23]:
df = deliveries.merge(matches, left_on='match_id', right_on='id', how='inner')

In [24]:
df.head()

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,...,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0.0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0.0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0.0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0.0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0.0,...,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen


In [26]:
batsman_match_runs = df.groupby(['match_id', 'date', 'venue', 'batting_team', 'bowling_team', 'batter'], as_index=False).agg(
    runs_scored=('batsman_runs', 'sum'),
    balls_faced=('ball', 'count'),
    is_dismissed=('player_dismissed', lambda x: (x.notna()).sum()) # approximation for dismissal
)

In [28]:
# Find the earliest over and ball each batter faced in every match
first_appearance = df.groupby(['match_id', 'batter'], as_index=False).agg(
    min_over=('over', 'min'),
    min_ball=('ball', 'min')
)

# Merge this back into our main batsman_match_runs table
batsman_match_runs = batsman_match_runs.merge(first_appearance, on=['match_id', 'batter'], how='left')

# Rank the batters by their entry time to determine their batting position
batsman_match_runs['batting_position'] = batsman_match_runs.groupby('match_id')['min_over'].rank(method='dense')

In [35]:
# Calculate overall historical player stats safely
player_overall = df.groupby('batter').agg(
    total_runs=('batsman_runs', 'sum'),
    total_balls=('ball', 'count'),
    total_dismissals=('player_dismissed', lambda x: x.notna().sum())
).reset_index()

# Compute Batting Average and Strike Rate
player_overall['batting_average'] = player_overall['total_runs'] / player_overall['total_dismissals'].replace(0, 1)
player_overall['strike_rate'] = (player_overall['total_runs'] / player_overall['total_balls'].replace(0, 1)) * 100

# Merge these historical stats back into our batsman_match_runs table
batsman_match_runs = batsman_match_runs.merge(
    player_overall[['batter', 'batting_average', 'strike_rate']],
    on='batter',
    how='left'
)

In [36]:
# 5. Opposition Bowling Strength (Average runs conceded per match by bowling team)
bowling_strength = df.groupby('bowling_team')['total_runs'].mean().reset_index().rename(columns={'total_runs': 'opp_bowling_strength'})
batsman_match_runs = batsman_match_runs.merge(bowling_strength, on='bowling_team', how='left')

# 6. Toss Result feature (1 if batting team won toss, else 0)
batsman_match_runs = batsman_match_runs.merge(matches[['id', 'toss_winner']], left_on='match_id', right_on='id', how='left')
batsman_match_runs['toss_won'] = (batsman_match_runs['batting_team'] == batsman_match_runs['toss_winner']).astype(int)

# Select final features and target for Model 1
model_data = batsman_match_runs[[
    'batting_average', 'strike_rate', 'opp_bowling_strength',
    'batting_position', 'toss_won', 'runs_scored', 'venue'
]].dropna()

print(model_data.head())

   batting_average  strike_rate  opp_bowling_strength  batting_position  \
0        32.666667   127.548807              1.282868               1.0   
1        28.388889   122.541966              1.282868               9.0   
2         8.000000    76.190476              1.282868              10.0   
3        13.000000    73.584906              1.282868               5.0   
4        27.131579   105.527124              1.282868               1.0   

   toss_won  runs_scored                  venue  
0         0        158.0  M Chinnaswamy Stadium  
1         0         12.0  M Chinnaswamy Stadium  
2         0          5.0  M Chinnaswamy Stadium  
3         0         20.0  M Chinnaswamy Stadium  
4         0         10.0  M Chinnaswamy Stadium  


In [37]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Define Features (X) and Target (y)
features = ['batting_average', 'strike_rate', 'opp_bowling_strength', 'batting_position', 'toss_won', 'venue']
X = model_data[features]
y = model_data['runs_scored']

# 2. Train-Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Preprocessor to handle categorical variable ('venue') using One-Hot Encoding
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['venue'])
    ],
    remainder='passthrough'
)

# 4. Create Pipeline combining Preprocessing and Multiple Linear Regression
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# 5. Fit the model on the training data
model.fit(X_train, y_train)

# 6. Predict on the test data and evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Multiple Linear Regression Results:")
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mse):.2f}")
print(f"R-squared (R2 Score): {r2:.4f}")

Multiple Linear Regression Results:
Root Mean Squared Error (RMSE): 17.76
R-squared (R2 Score): 0.1825


In [38]:
import os

# Create the models directory if it doesn't already exist
os.makedirs('models', exist_ok=True)

import pickle

# Save the trained model pipeline to disk
model_filename = 'models/model_1_linear.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(model, file)

print(f"Model successfully saved to {model_filename}")

Model successfully saved to models/model_1_linear.pkl


In [39]:
from sklearn.preprocessing import PolynomialFeatures

# 1. Create a Pipeline that includes Polynomial Features
# degree=2 adds squared terms (e.g., batting_average^2, strike_rate^2) and interaction terms
poly_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('regressor', LinearRegression())
])

# 2. Fit the Polynomial Model on the training data
poly_model.fit(X_train, y_train)

# 3. Predict and evaluate
y_pred_poly = poly_model.predict(X_test)
mse_poly = mean_squared_error(y_test, y_pred_poly)
r2_poly = r2_score(y_test, y_pred_poly)

print(f"Polynomial Regression Results:")
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mse_poly):.2f}")
print(f"R-squared (R2 Score): {r2_poly:.4f}")

# 4. Save Model 2 to disk
os.makedirs('models', exist_ok=True)
poly_filename = 'models/model_2_polynomial.pkl'
with open(poly_filename, 'wb') as file:
    pickle.dump(poly_model, file)

print(f"Model successfully saved to {poly_filename}")

Polynomial Regression Results:
Root Mean Squared Error (RMSE): 17.74
R-squared (R2 Score): 0.1847
Model successfully saved to models/model_2_polynomial.pkl


In [41]:
from sklearn.tree import DecisionTreeRegressor

# 1. Create a Pipeline with a regularized Decision Tree (limiting depth to prevent overfitting)
tree_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', DecisionTreeRegressor(max_depth=5, min_samples_leaf=10, random_state=42))
])

# 2. Fit the model on the training data
tree_model.fit(X_train, y_train)

# 3. Predict and evaluate
y_pred_tree = tree_model.predict(X_test)
mse_tree = mean_squared_error(y_test, y_pred_tree)
r2_tree = r2_score(y_test, y_pred_tree)

print(f"Tuned Decision Tree Regressor Results:")
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mse_tree):.2f}")
print(f"R-squared (R2 Score): {r2_tree:.4f}")

# 4. Save Model 3 to disk
tree_filename = 'models/model_3_decision_tree.pkl'
with open(tree_filename, 'wb') as file:
    pickle.dump(tree_model, file)

print(f"Model successfully saved to {tree_filename}")

Tuned Decision Tree Regressor Results:
Root Mean Squared Error (RMSE): 18.36
R-squared (R2 Score): 0.1272
Model successfully saved to models/model_3_decision_tree.pkl


In [42]:
from sklearn.ensemble import RandomForestRegressor

# 1. Create a Pipeline with a Random Forest Regressor
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42))
])

# 2. Fit the model on the training data
rf_model.fit(X_train, y_train)

# 3. Predict and evaluate
y_pred_rf = rf_model.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest Regressor Results:")
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mse_rf):.2f}")
print(f"R-squared (R2 Score): {r2_rf:.4f}")

# 4. Save Model 4 to disk
rf_filename = 'models/model_4_random_forest.pkl'
with open(rf_filename, 'wb') as file:
    pickle.dump(rf_model, file)

print(f"Model successfully saved to {rf_filename}")

Random Forest Regressor Results:
Root Mean Squared Error (RMSE): 18.06
R-squared (R2 Score): 0.1554
Model successfully saved to models/model_4_random_forest.pkl


In [43]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Create results directory
os.makedirs('results', exist_ok=True)

# 1. Compile model metrics into a DataFrame
results_df = pd.DataFrame({
    'Model': ['Multiple Linear Regression', 'Polynomial Regression', 'Decision Tree Regressor', 'Random Forest Regressor'],
    'RMSE': [np.sqrt(mse), np.sqrt(mse_poly), np.sqrt(mse_tree), np.sqrt(mse_rf)],
    'R2 Score': [r2, r2_poly, r2_tree, r2_rf]
})

# 2. Plot and save Comparison Table as an image
fig, ax = plt.subplots(figsize=(8, 3))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=results_df.values, colLabels=results_df.columns, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.2)

plt.title("Model Performance Comparison", fontsize=14, weight='bold', pad=20)
plt.savefig('results/comparison_table.png', bbox_inches='tight', dpi=300)
plt.close()

# 3. Extract and plot Feature Importance from the Random Forest model
# Get feature names from preprocessor and features list
cat_encoder = rf_model.named_steps['preprocessor'].named_transformers_['cat']
encoded_venue_features = list(cat_encoder.get_feature_names_out(['venue']))
all_features = encoded_venue_features + [f for f in features if f != 'venue']

importances = rf_model.named_steps['regressor'].feature_importances_

# Sort importances for better plotting
indices = np.argsort(importances)[-10:]  # Top 10 features

plt.figure(figsize=(10, 6))
plt.title("Top Feature Importances (Random Forest)", fontsize=14, weight='bold')
plt.barh(range(len(indices)), importances[indices], color='skyblue', align='center')
plt.yticks(range(len(indices)), [all_features[i] for i in indices])
plt.xlabel("Relative Importance", fontsize=12)
plt.tight_layout()
plt.savefig('results/feature_importance.png', dpi=300)
plt.close()

print("Comparison table and feature importance chart successfully generated and saved in 'results/'!")

Comparison table and feature importance chart successfully generated and saved in 'results/'!
